## Qubit characterization notebook

This is an example notebook that run through the standard qubit characterization experiments.

In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.use_mathtext'] = False

%matplotlib widget
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from acadia_qmsmt.utils import load_yaml

try: # if using the new qcodes based instrument server
    from instrumentserver.client import ClientStation  
    instrument_station = ClientStation(host='localhost', timeout=600, init_instruments = None)
except Exception as e: # make a fallback dummy object
    instrument_station =  type('Dummy', (), {})()
    instrument_station.save_parameters = lambda *args:  None


# a simple python file that stores global settings
# this currently uses a very simple relative path, remember to change this if you move the modules around
from system_info import DATA_DIR, BOARD_IP, YAML_PATH



#### Example of controlling instruments:

In [ ]:
# print(instrument_station.instruments)
# instrument_station["sc_spa_lincA"].power()

# Readout Spec

In [ ]:
from acadia_qmsmt.runtimes.resonator_spectroscopy import  ResonatorSpectroscopyRuntime

freqs = np.linspace(-5e6, 5e6, 101) + 9.309e9
iterations = 500
run_delay = 2000
rt = ResonatorSpectroscopyRuntime(stimulus="ro_stimulus", capture="ro_capture", frequencies=freqs, iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Readout_Spec/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Qubit Spec

In [ ]:
from acadia_qmsmt.runtimes.qubit_spectroscopy import  QubitSpectroscopyRuntime

qubit_center_freq = 8.0477e9

qubit_frequencies = np.linspace(-5e6, 5e6, 201) + qubit_center_freq

# can either be a name of a pulse in the config dict or a dict that specifies the pulse paramters
saturation_pulse_config = {
    "name": "saturation_pulse",
    "data": "hann",
    "flat": 10e-6,
    "ramp": 200e-9,
    "use_stretch": True,
    "scale": 0.01
}


iterations = 200
run_delay = 200e3
# capture_window_name = "matched"
capture_window_name = "boxcar"
rt = QubitSpectroscopyRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture",
                              qubit_frequencies=qubit_frequencies,
                              saturation_pulse_config=saturation_pulse_config,
                              capture_window_name = capture_window_name,
                              iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP,local_directory=DATA_DIR + "Qubit_Spec/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Qubit Rabi

In [ ]:
from acadia_qmsmt.runtimes.qubit_amplitude_rabi import  QubitPulseAmplitudeCalibrationRuntime

cal_selective = False
iterations = 2000
run_delay = 300e3
capture_window_name = "matched"
# capture_window_name = "boxcar"

if cal_selective:
    qubit_pulse_name = "R_x_180_selective"
    qubit_amplitudes = np.linspace(-0.15, 0.15, 51)
else:
    qubit_pulse_name = "R_x_180"
    qubit_amplitudes = np.linspace(-0.9, 0.9, 51)

rt = QubitPulseAmplitudeCalibrationRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture",
                                  qubit_amplitudes=qubit_amplitudes,capture_window_name=capture_window_name,
                                  qubit_pulse_name=qubit_pulse_name, readout_pulse_name="readout",
                                  iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Qubit_Rabi/%y%m%d/%H%M%S")
rt.display()

print(rt.local_directory)


# Ro window calibration

In [ ]:
from acadia_qmsmt.runtimes.readout_window_calibration import  ReadoutWindowCalibrationRuntime

iterations = 20000
qubit_pulse_name = "R_x_180"
rt = ReadoutWindowCalibrationRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture",
                            qubit_pulse_name=qubit_pulse_name,
                           readout_pulse_name="readout",
                           capture_memory_name="readout_trace",
                           iterations=iterations, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Readout_WindowCal/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Readout Histogram & Metrics (Fidelity, etc.)

In [ ]:
from acadia_qmsmt.runtimes.readout_fidelity_metrics import  ReadoutFidelityRuntime

iterations = 20000
run_delay = 400e3
capture_window_name = "matched"

num_prep_rounds = 0

qubit_pulse_name = "R_x_180"
rt = ReadoutFidelityRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture",
                            qubit_pulse_name=qubit_pulse_name,
                            capture_window_name=capture_window_name,
                            iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH, num_prep_rounds=num_prep_rounds)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Readout_Hist_and_Metrics/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Qubit T1

In [ ]:
from acadia_qmsmt.runtimes.qubit_t1 import  QubitRelaxationRuntime

delay_times = np.linspace(0, 300e-6, 41) + 800e-9
iterations = 5000
run_delay = 500e3
capture_window_name = "matched"
# qubit_pulse_name = "R_x_180_selective"

qubit_pulse_name = "R_x_180"
rt = QubitRelaxationRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture",
                            delay_times=delay_times,
                            qubit_pulse_name=qubit_pulse_name,
                            capture_window_name=capture_window_name,
                            iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Qubit_T1/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Qubit T2

In [ ]:
from acadia_qmsmt.runtimes.qubit_t2 import  QubitCoherenceRuntime

delay_times = np.linspace(0, 80e-6, 101) + 600e-9
pulse_type = ""
qubit_pi_pulse_name = "R_x_180" + pulse_type


do_echo = True

virtual_detuning = -0.1e6
iterations = 5000
run_delay = 500e3
capture_window_name = "matched"
rt = QubitCoherenceRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture",
                           delay_times=delay_times, virtual_detuning=virtual_detuning, do_echo=do_echo,
                           qubit_pi_pulse_name=qubit_pi_pulse_name,
                           capture_window_name=capture_window_name,
                           iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"Qubit_T2{'E' if rt.do_echo else 'R'}/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# RO Spec with qubit prep  (rough $\chi$ msmt)

In [ ]:
from acadia_qmsmt.runtimes.resonator_spectroscopy_prepQubit import  ResonatorSpectroscopyPrepQubitRuntime

freqs = np.linspace(-4e6, 4e6, 101) + load_yaml(YAML_PATH)["ro_stimulus"]["channel_config"]["nco_frequency"]
qubit_pulse_name = "R_x_180"
iterations = 500
run_delay = 2000
rt = ResonatorSpectroscopyPrepQubitRuntime(stimulus="ro_stimulus", capture="ro_capture", qubit_stimulus="qb_stimulus",  
                                        frequencies=freqs, qubit_pulse_name=qubit_pulse_name, 
                                        iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Readout_Spec_prep_qubit/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Qubit ef spec

In [ ]:
from acadia_qmsmt.runtimes.qubit_anharmonicity import  QubitAnharmonicityRuntime

detune_frequencies = np.linspace(-170e6, -150e6, 101) 


ef_pulse_config = {
    "name": "saturation_pulse",
    "data": "hann",
    "flat": 10e-6,
    "ramp": 100e-9,
    "use_stretch": False, # !!Note: must be False for numeric detune
    "scale": 0.01
}

qubit_pulse_name = "R_x_180"

iterations = 3000
run_delay = 200e3 #ns
capture_window_name = "matched"

rt = QubitAnharmonicityRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", 
                               qubit_pulse_name=qubit_pulse_name, 
                               ef_pulse_config=ef_pulse_config, 
                               detune_frequencies=detune_frequencies,capture_window_name=capture_window_name,
                               iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Qubit_efSpec/%y%m%d/%H%M%S")
rt.display()

##  Qubit ef Rabi

In [ ]:
from acadia_qmsmt.runtimes.qubit_ef_amplitude_rabi import  QubitEFPulseAmplitudeCalibrationRuntime

qubit_amplitudes = np.linspace(-0.6, 0.6, 51)

cal_selective = False
iterations = 2000
run_delay = 300e3

qubit_ge_pulse_name = "R_x_180"
qubit_ef_pulse_name = "R_x_180_ef"
capture_window_name = "matched"

rt = QubitEFPulseAmplitudeCalibrationRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", 
                                             qubit_amplitudes = qubit_amplitudes,
                               qubit_ge_pulse_name=qubit_ge_pulse_name, qubit_ef_pulse_name=qubit_ef_pulse_name,
                               capture_window_name=capture_window_name,
                               iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Qubit_efRabi/%y%m%d/%H%M%S")
rt.display()

# Qubit RPM

In [ ]:
from acadia_qmsmt.runtimes.qubit_rabi_population_msmt import QubitRpmRuntime

qubit_amplitudes = np.linspace(-0.8, 0.8, 51)

cal_selective = False
iterations = 50000
run_delay = 300e3

qubit_ge_pulse_name = "R_x_180"
qubit_ef_pulse_name = "R_x_180_ef"
capture_window_name = "matched"

rt = QubitRpmRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture",
                    qubit_amplitudes = qubit_amplitudes,
                    qubit_ge_pulse_name=qubit_ge_pulse_name, qubit_ef_pulse_name=qubit_ef_pulse_name,
                    capture_window_name=capture_window_name,
                    iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Qubit_RPM/%y%m%d/%H%M%S")
rt.display()

# Qubit RB

In [ ]:
from acadia_qmsmt.runtimes.qubit_RB import  QubitRBRuntime

qubit_pi_pulse_name = "R_x_180"

seq_lengths =  np.arange(0,400,10)  # np.linspace(0, 500, 21, dtype=int) # Number of pulses in the sequence

print(seq_lengths)

iterations = 10000
run_delay = 500e3
capture_window_name = "matched"
rt = QubitRBRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture",
                    seq_lengths=seq_lengths,
                           iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + f"Qubit_RB/%y%m%d/%H%M%S")
rt.display()
_ = instrument_station.save_parameters(rt.local_directory + "/inst_params.json")

# Other codes

### Qubit length rabi, to show length sweep with register

In [ ]:
from acadia_qmsmt.runtimes.qubit_length_rabi import  QubitLengthRabiRuntime


iterations = 2000
run_delay = 300e3
capture_window_name = "matched"
# capture_window_name = "boxcar"

qubit_pulse_name = "R_x_180"
flat_length_list = np.linspace(0, 10e-6, 501) + 10e-9

rt = QubitLengthRabiRuntime(qubit_stimulus="qb_stimulus", readout_stimulus="ro_stimulus", readout_capture="ro_capture", 
                                  flat_length_list=flat_length_list,capture_window_name=capture_window_name,
                                  qubit_pulse_name=qubit_pulse_name, readout_pulse_name="readout",
                                  iterations=iterations, run_delay=run_delay, yaml_path=YAML_PATH)
rt.deploy(BOARD_IP, local_directory=DATA_DIR + "Qubit_LengthRabi/%y%m%d/%H%M%S")
rt.display()
